In [1]:
#llm을 통해 QA수집
from utils import generate_rag_dataset
generate_rag_dataset(
        input_path="sunic_all_pages.txt", output_dir="rag_data3",BATCH_SIZE=3

    )

작업 시작: 13개의 묶음 확인됨 (저장 경로: rag_data3)

[시도 1/13] 분석 중...
-> 완료! [이번 추출: 89개]

[시도 2/13] 분석 중...
-> 완료! [이번 추출: 103개]

[시도 3/13] 분석 중...
-> 완료! [이번 추출: 86개]

[시도 4/13] 분석 중...
-> 완료! [이번 추출: 100개]

[시도 5/13] 분석 중...
-> 완료! [이번 추출: 102개]

[시도 6/13] 분석 중...
-> 완료! [이번 추출: 106개]

[시도 7/13] 분석 중...
-> 완료! [이번 추출: 95개]

[시도 8/13] 분석 중...
-> [오류] 묶음 8 처리 중 (시도 1/3): 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
-> 서버가 바쁩니다. 30초 대기 후 재시도합니다.
-> 완료! [이번 추출: 110개]

[시도 9/13] 분석 중...
-> 완료! [이번 추출: 82개]

[시도 10/13] 분석 중...
-> 완료! [이번 추출: 148개]

[시도 11/13] 분석 중...
-> 완료! [이번 추출: 137개]

[시도 12/13] 분석 중...
-> 완료! [이번 추출: 101개]

[시도 13/13] 분석 중...
-> 완료! [이번 추출: 109개]

최종 완료!


In [4]:
#QA json 통합
from utils import merge_json_files

merge_json_files()

총 2개의 하위 폴더를 찾았습니다: ['rag_data3', 'rag_data3_ko']

[rag_data3] 폴더 병합 시작...
 -> 완료! 총 1368개 데이터 통합 -> 'data\rag_data3\merged_rag_data3.json'

[rag_data3_ko] 폴더 병합 시작...
 -> 완료! 총 1388개 데이터 통합 -> 'data\rag_data3_ko\merged_rag_data3_ko.json'

모든 폴더의 병합 작업이 최종 완료되었습니다!


In [6]:
from utils import augment_large_dataset

augment_large_dataset("data/rag_data3", "merged_rag_data3.json", BATCH_SIZE=40)

-> 처음부터 증강을 시작합니다. 총 데이터: 1368개

[1/35 묶음] 0번 ~ 39번 처리 중...

[2/35 묶음] 40번 ~ 79번 처리 중...

[3/35 묶음] 80번 ~ 119번 처리 중...

[4/35 묶음] 120번 ~ 159번 처리 중...

[5/35 묶음] 160번 ~ 199번 처리 중...

[6/35 묶음] 200번 ~ 239번 처리 중...

[7/35 묶음] 240번 ~ 279번 처리 중...

[8/35 묶음] 280번 ~ 319번 처리 중...

[9/35 묶음] 320번 ~ 359번 처리 중...

[10/35 묶음] 360번 ~ 399번 처리 중...

[11/35 묶음] 400번 ~ 439번 처리 중...

[12/35 묶음] 440번 ~ 479번 처리 중...

[13/35 묶음] 480번 ~ 519번 처리 중...

[14/35 묶음] 520번 ~ 559번 처리 중...

[15/35 묶음] 560번 ~ 599번 처리 중...

[16/35 묶음] 600번 ~ 639번 처리 중...

[17/35 묶음] 640번 ~ 679번 처리 중...

[18/35 묶음] 680번 ~ 719번 처리 중...

[19/35 묶음] 720번 ~ 759번 처리 중...

[20/35 묶음] 760번 ~ 799번 처리 중...

[21/35 묶음] 800번 ~ 839번 처리 중...

[22/35 묶음] 840번 ~ 879번 처리 중...

[23/35 묶음] 880번 ~ 919번 처리 중...

[24/35 묶음] 920번 ~ 959번 처리 중...

[25/35 묶음] 960번 ~ 999번 처리 중...

[26/35 묶음] 1000번 ~ 1039번 처리 중...

[27/35 묶음] 1040번 ~ 1079번 처리 중...

[28/35 묶음] 1080번 ~ 1119번 처리 중...

[29/35 묶음] 1120번 ~ 1159번 처리 중...

[30/35 묶음] 1160번 ~ 1199번 처리 중...

[31/35 묶음] 1

In [8]:
import json
import re

with open("data/rag_data3/merged_rag_data3.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"총 데이터 개수: {len(data)}")
foreign_count = 0
for idx, item in enumerate(data):
    q = item.get("question", "")
    a = item.get("answer", "")
    combined = q + a
    
    is_foreign = False
    
    # 1. 중국어 한자 검사 
    if re.search(r'[\u4e00-\u9fff]|[\u3400-\u4dbf]', combined):
        is_foreign = True
        lang_type = "중국어"
    
    # 2. 영어 문장 검사 
    elif not re.search(r'[\uac00-\ud7a3]', combined) and re.search(r'[a-zA-Z]{5,}', combined):
        is_foreign = True
        lang_type = "영어"

    if is_foreign:
        foreign_count += 1
        if foreign_count <=7:  
            print(f"[{lang_type} 데이터 발견 - 인덱스 {idx}]")
            print(f"Q: {q}")
            print(f"A: {a[:150]}..." if len(a) > 150 else f"A: {a}") # 답변 너무 길면 잘라서 출력
            print("-" * 40)

print(f"\n 한글을 제외한 데이터는 총 {foreign_count}개입니다.")

총 데이터 개수: 1368
[중국어 데이터 발견 - 인덱스 40]
Q: 선익시스템은 2024년 몇 월에 BOE(중)와 어떤 규모의 OLED 관련 계약을 체결했습니까?
A: 선익시스템은 2024년 6월에 BOE(중)와 8.6G OLED 用 관련 계약을 체결했습니다.
----------------------------------------
[중국어 데이터 발견 - 인덱스 45]
Q: 선익시스템은 2020년 몇 월에 BOE(중)와 어떤 규모의 Micro OLED 관련 계약을 체결했습니까?
A: 선익시스템은 2020년 4월에 BOE(중)와 300mm Micro OLED 用 관련 계약을 체결했습니다.
----------------------------------------
[중국어 데이터 발견 - 인덱스 46]
Q: 선익시스템은 2018년 몇 월에 SeeYA(중)와 어떤 규모의 Micro OLED 관련 계약을 체결했습니까?
A: 선익시스템은 2018년 1월에 SeeYA(중)와 300mm Micro OLED 用 관련 계약을 체결했습니다.
----------------------------------------
[중국어 데이터 발견 - 인덱스 203]
Q: 선익시스템이 말하는 '선익 人'은 어떤 인재를 지칭하나요?
A: 선익시스템이 말하는 '선익 人'은 신뢰와 전문성을 바탕으로 도전과 혁신을 두려워하지 않는 단정하고 매너 있는 인재를 지칭합니다.
----------------------------------------
[중국어 데이터 발견 - 인덱스 278]
Q: Sunic system拥有多少年以上研究OLED的专业人才？
A: Sunic system拥有研究OLED15年以上的专业人才。
----------------------------------------
[중국어 데이터 발견 - 인덱스 279]
Q: Sunic system的OLED专业工程师比例保持在多少以上？
A: Sunic system的OLED专业工程师比例保持在80%以上。
----------------------------